In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step6 GO KEGG enrichment
# 初始化
# ============================================================

import os
import pandas as pd
import numpy as np


project_dir = _hcc_path('')


step5_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step5_DEG"
)


step6_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step6_Enrichment"
)


go_dir = os.path.join(
    step6_dir,
    "GO_Result"
)


kegg_dir = os.path.join(
    step6_dir,
    "KEGG_Result"
)


fig_dir = os.path.join(
    step6_dir,
    "Figures"
)


for d in [
    step6_dir,
    go_dir,
    kegg_dir,
    fig_dir
]:
    os.makedirs(
        d,
        exist_ok=True
    )


print(step6_dir)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
import os
import pandas as pd
import numpy as np


project_dir = _hcc_path('')


step5_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step5_DEG"
)


step6_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step6_Enrichment"
)


go_dir = os.path.join(
    step6_dir,
    "GO_Result"
)


fig_dir = os.path.join(
    step6_dir,
    "Figures"
)


print("Environment loaded")

In [ ]:
deg_file = os.path.join(
    step5_dir,
    "DEG_Result",
    "LIHC_Tumor_vs_Normal_DEG_All.csv"
)


deg = pd.read_csv(
    deg_file
)


print(
    deg.shape
)


deg.head()

In [ ]:
up_genes = deg.loc[
    deg["Significance"]=="Up",
    "Gene"
].tolist()


down_genes = deg.loc[
    deg["Significance"]=="Down",
    "Gene"
].tolist()


print(
    "Up:",
    len(up_genes)
)


print(
    "Down:",
    len(down_genes)
)

In [ ]:
import gseapy as gp


go_up = gp.enrichr(

    gene_list=up_genes,

    gene_sets=[
        "GO_Biological_Process_2023"
    ],

    organism="human",

    outdir=None

)


print("GO finished")

In [ ]:
go_up_results = go_up.results


print(
    go_up_results.shape
)


go_up_results.head()

In [ ]:
# ============================================================
# Step6 GO dotplot
# Up genes
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# 取FDR最低的Top15

go_dot = (
    go_up_results
    .sort_values(
        by="Adjusted P-value",
        ascending=True
    )
    .head(15)
    .copy()
)


print(
    go_dot[[
        "Term",
        "Adjusted P-value",
        "Overlap"
    ]]
)

In [ ]:
# ============================================================
# Gene ratio计算
# ============================================================


go_dot["GeneRatio"] = (
    go_dot["Overlap"]
    .str.split("/")
    .apply(
        lambda x:
        int(x[0]) / int(x[1])
    )
)


go_dot.head()

In [ ]:
# ============================================================
# GO dotplot 保存版
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import os


plt.figure(
    figsize=(9,7)
)


plt.scatter(
    go_dot["GeneRatio"],
    range(len(go_dot)),
    s=(
        -np.log10(
            go_dot["Adjusted P-value"]
        )
        *80
    )
)


plt.yticks(
    range(len(go_dot)),
    go_dot["Term"]
)


plt.xlabel(
    "Gene Ratio"
)


plt.ylabel(
    "GO Biological Process"
)


plt.title(
    "GO Enrichment Analysis\nUpregulated Genes in LIHC"
)


plt.gca().invert_yaxis()


plt.tight_layout()


# ==========================
# 先保存
# ==========================

go_png = os.path.join(
    fig_dir,
    "GO_BP_Up_Dotplot.png"
)


plt.savefig(
    go_png,
    dpi=600,
    bbox_inches="tight"
)


print("Saved:")
print(go_png)


# 最后显示

plt.show()

In [ ]:
go_svg = os.path.join(
    fig_dir,
    "GO_BP_Up_Dotplot.svg"
)


plt.savefig(
    go_svg,
    bbox_inches="tight"
)

In [ ]:
# ============================================================
# Step6.12 GO Biological Process
# Down genes
# ============================================================


import gseapy as gp


go_down = gp.enrichr(

    gene_list=down_genes,

    gene_sets=[
        "GO_Biological_Process_2023"
    ],

    organism="human",

    outdir=None

)


print("GO Down finished")

In [ ]:
go_down_results = go_down.results


print(
    go_down_results.shape
)


go_down_results.head()

In [ ]:
go_down_results.to_csv(
    os.path.join(
        go_dir,
        "GO_BP_Down_genes.csv"
    ),
    index=False
)


print("GO Down saved")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


go_down_dot = (
    go_down_results
    .sort_values(
        by="Adjusted P-value",
        ascending=True
    )
    .head(15)
    .copy()
)


# Gene Ratio

go_down_dot["GeneRatio"] = (
    go_down_dot["Overlap"]
    .str.split("/")
    .apply(
        lambda x:
        int(x[0]) / int(x[1])
    )
)

In [ ]:
plt.figure(
    figsize=(9,7)
)


plt.scatter(
    go_down_dot["GeneRatio"],
    range(len(go_down_dot)),
    s=(
        -np.log10(
            go_down_dot["Adjusted P-value"]
        )
        *80
    )
)


plt.yticks(
    range(len(go_down_dot)),
    go_down_dot["Term"]
)


plt.xlabel(
    "Gene Ratio"
)


plt.ylabel(
    "GO Biological Process"
)


plt.title(
    "GO Enrichment Analysis\nDownregulated Genes in LIHC"
)


plt.gca().invert_yaxis()


plt.tight_layout()


# 保存（注意先保存）

plt.savefig(
    os.path.join(
        fig_dir,
        "GO_BP_Down_Dotplot.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ============================================================
# Step6.14 KEGG enrichment
# Up genes
# ============================================================

import gseapy as gp


kegg_up = gp.enrichr(

    gene_list=up_genes,

    gene_sets=[
        "KEGG_2021_Human"
    ],

    organism="human",

    outdir=None

)


print("KEGG Up finished")

In [ ]:
kegg_up_results = kegg_up.results


print(
    kegg_up_results.shape
)


kegg_up_results.head()

In [ ]:
kegg_up_results.to_csv(
    os.path.join(
        kegg_dir,
        "KEGG_Up_genes.csv"
    ),
    index=False
)


print("KEGG Up saved")

In [ ]:
# ============================================================
# KEGG Down genes
# ============================================================


kegg_down = gp.enrichr(

    gene_list=down_genes,

    gene_sets=[
        "KEGG_2021_Human"
    ],

    organism="human",

    outdir=None

)


print("KEGG Down finished")

In [ ]:
kegg_down_results = kegg_down.results


print(
    kegg_down_results.shape
)


kegg_down_results.head()

In [ ]:
kegg_down_results.to_csv(
    os.path.join(
        kegg_dir,
        "KEGG_Down_genes.csv"
    ),
    index=False
)


print("KEGG Down saved")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


kegg_plot = (
    kegg_up_results
    .sort_values(
        by="Adjusted P-value",
        ascending=True
    )
    .head(15)
    .copy()
)


kegg_plot["GeneRatio"] = (
    kegg_plot["Overlap"]
    .str.split("/")
    .apply(
        lambda x:
        int(x[0]) / int(x[1])
    )
)

In [ ]:
plt.figure(
    figsize=(9,7)
)


plt.scatter(
    kegg_plot["GeneRatio"],
    range(len(kegg_plot)),
    s=(
        -np.log10(
            kegg_plot["Adjusted P-value"]
        )
        *80
    )
)


plt.yticks(
    range(len(kegg_plot)),
    kegg_plot["Term"]
)


plt.xlabel(
    "Gene Ratio"
)


plt.ylabel(
    "KEGG Pathway"
)


plt.title(
    "KEGG Enrichment\nUpregulated Genes in LIHC"
)


plt.gca().invert_yaxis()


plt.tight_layout()


plt.savefig(
    os.path.join(
        fig_dir,
        "KEGG_Up_Dotplot.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
# ============================================================
# Step6.16 KEGG Down genes
# ============================================================

import gseapy as gp


kegg_down = gp.enrichr(

    gene_list=down_genes,

    gene_sets=[
        "KEGG_2021_Human"
    ],

    organism="human",

    outdir=None

)


print("KEGG Down finished")

In [ ]:
# ============================================================
# 获取KEGG Down结果
# ============================================================

kegg_down_results = kegg_down.results


print(
    kegg_down_results.shape
)


kegg_down_results.head()

In [ ]:
# ============================================================
# 保存KEGG Down
# ============================================================

kegg_down_results.to_csv(
    os.path.join(
        kegg_dir,
        "KEGG_Down_genes.csv"
    ),
    index=False
)


print("KEGG Down saved")

In [ ]:
# ============================================================
# KEGG Down dotplot
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


kegg_down_plot = (
    kegg_down_results
    .sort_values(
        by="Adjusted P-value",
        ascending=True
    )
    .head(15)
    .copy()
)


# Gene Ratio

kegg_down_plot["GeneRatio"] = (
    kegg_down_plot["Overlap"]
    .str.split("/")
    .apply(
        lambda x:
        int(x[0]) / int(x[1])
    )
)


plt.figure(
    figsize=(9,7)
)


plt.scatter(
    kegg_down_plot["GeneRatio"],
    range(len(kegg_down_plot)),
    s=(
        -np.log10(
            kegg_down_plot["Adjusted P-value"]
        )
        *80
    )
)


plt.yticks(
    range(len(kegg_down_plot)),
    kegg_down_plot["Term"]
)


plt.xlabel(
    "Gene Ratio"
)


plt.ylabel(
    "KEGG Pathway"
)


plt.title(
    "KEGG Enrichment\nDownregulated Genes in LIHC"
)


plt.gca().invert_yaxis()


plt.tight_layout()


plt.savefig(
    os.path.join(
        fig_dir,
        "KEGG_Down_Dotplot.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
kegg_down_results.head()

In [ ]:
# ============================================================
# Step6 Final save
# ============================================================

# GO

go_up_results.to_csv(
    os.path.join(
        go_dir,
        "GO_BP_Up_results.csv"
    ),
    index=False
)


go_down_results.to_csv(
    os.path.join(
        go_dir,
        "GO_BP_Down_results.csv"
    ),
    index=False
)



# KEGG

kegg_up_results.to_csv(
    os.path.join(
        kegg_dir,
        "KEGG_Up_results.csv"
    ),
    index=False
)


kegg_down_results.to_csv(
    os.path.join(
        kegg_dir,
        "KEGG_Down_results.csv"
    ),
    index=False
)


print("All enrichment tables saved")